### results_1872_2024

In [31]:
# ==========================================================
# Analizaremos la estructura general del dataset de resultados
# históricos para identificar valores nulos y duplicados.
# ==========================================================

import pandas as pd

results = pd.read_csv("../data/raw/results_1872_2024/results.csv")

print("Dimensiones:", results.shape)
print("\nTipos de datos:")
display(results.dtypes)

print("\nValores nulos:")
display(results.isnull().sum())

print("\nDuplicados:", results.duplicated().sum())

Dimensiones: (49502, 9)

Tipos de datos:


date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
neutral          bool
dtype: object


Valores nulos:


date          0
home_team     0
away_team     0
home_score    7
away_score    7
tournament    0
city          0
country       0
neutral       0
dtype: int64


Duplicados: 0


In [32]:
# ==========================================================
# Analizaremos los registros con marcadores faltantes para
# decidir si deben eliminarse del dataset.
# ==========================================================

display(
    results[
        results["home_score"].isna() |
        results["away_score"].isna()
    ]
)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49495,2026-07-05,Brazil,Norway,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49496,2026-07-05,Mexico,England,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
49497,2026-07-06,Portugal,Spain,NaN,NaN,FIFA World Cup,Dallas,United States,True
49498,2026-07-06,United States,Belgium,NaN,NaN,FIFA World Cup,Seattle,United States,False
49499,2026-07-06,Argentina,Egypt,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49500,2026-07-06,Switzerland,Colombia,NaN,NaN,FIFA World Cup,Vancouver,Canada,True
49501,2026-07-09,France,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True


In [33]:
# ==========================================================
# Eliminaremos los partidos sin resultado, ya que corresponden
# a encuentros futuros del Mundial 2026.
# ==========================================================

results = results.dropna(
    subset=["home_score", "away_score"]
)

print("Dimensiones:", results.shape)
print("\nValores nulos:")
display(results.isnull().sum())

Dimensiones: (49495, 9)

Valores nulos:


date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

## Results Dataset

Este dataset contiene el historial de partidos internacionales de selecciones nacionales, incluyendo la fecha, equipos participantes, marcador, torneo y sede del encuentro.

Durante el preprocesamiento se realizaron las siguientes tareas:

- Se verificó la ausencia de registros duplicados.
- Se eliminaron los partidos sin resultado registrado, correspondientes a encuentros futuros del Mundial FIFA 2026.
- Se comprobó que el resto de las variables no presentaran valores nulos.

El resultado es un dataset limpio y consistente con los resultados históricos disponibles, listo para integrarse con los demás datasets del proyecto y utilizarse en el entrenamiento del modelo predictor del Mundial FIFA 2026.

### Seguimos en results_1872_2024

In [35]:
import pandas as pd

goalscorers = pd.read_csv("../data/raw/results_1872_2024/goalscorers.csv")
shootouts = pd.read_csv("../data/raw/results_1872_2024/shootouts.csv")
former_names = pd.read_csv("../data/raw/results_1872_2024/former_names.csv")

print("GOALSCORERS")
print(goalscorers.shape)
display(goalscorers.head())

print("\nSHOOTOUTS")
print(shootouts.shape)
display(shootouts.head())

print("\nFORMER_NAMES")
print(former_names.shape)
display(former_names.head())

GOALSCORERS
(47867, 8)


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,False,False
3,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75.0,False,False
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,False,False



SHOOTOUTS
(681, 5)


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN



FORMER_NAMES
(36, 4)


,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04
2,Curaçao,Netherlands Antilles,1957-03-03,2010-10-10
3,Czechoslovakia,Bohemia,1903-04-05,1919-01-01
4,Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01


### Integración del dataset de resultados históricos

La carpeta `results_1872_2024` contiene cuatro archivos relacionados con los partidos internacionales de selecciones nacionales. Sin embargo, no todos poseen el mismo nivel de granularidad, por lo que no es conveniente unirlos directamente.

Se tomó como dataset principal `results.csv`, ya que cada fila representa un partido y constituye la unidad de análisis utilizada para entrenar el modelo de predicción.

Durante el proceso de integración se siguió el siguiente criterio:

- **`results.csv`** se utilizó como base del dataset final.
- **`shootouts.csv`** se integró al dataset principal para incorporar información sobre los partidos definidos mediante tanda de penales.
- **`former_names.csv`** se empleó únicamente para normalizar nombres históricos de selecciones nacionales, garantizando la consistencia de los datos.
- **`goalscorers.csv`** no se integró al dataset final debido a que registra información a nivel de cada gol anotado (una fila por gol), mientras que `results.csv` registra información a nivel de partido (una fila por encuentro). Unir ambos archivos directamente produciría duplicación de registros y alteraría la estructura del dataset.

In [36]:
# ==========================================================
# Integraremos la información de las tandas de penales al
# dataset principal de resultados históricos.
# ==========================================================

# Indicador de definición por penales
shootouts["went_to_shootout"] = 1

# Seleccionar únicamente las columnas necesarias
shootouts_merge = shootouts[
    ["date", "home_team", "away_team", "winner", "went_to_shootout"]
]

# Merge con el dataset principal
results = results.merge(
    shootouts_merge,
    on=["date", "home_team", "away_team"],
    how="left"
)

# Completar partidos sin tanda de penales
results["went_to_shootout"] = results["went_to_shootout"].fillna(0).astype(int)
results["winner"] = results["winner"].fillna("No Shootout")

print(results.shape)
display(results.head())

(49495, 11)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,winner,went_to_shootout
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,No Shootout,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,No Shootout,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,No Shootout,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,No Shootout,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,No Shootout,0


In [37]:
# ==========================================================
# Normalizaremos los nombres históricos de las selecciones
# nacionales utilizando el archivo former_names.csv.
# ==========================================================

former_names["start_date"] = pd.to_datetime(former_names["start_date"])
former_names["end_date"] = pd.to_datetime(former_names["end_date"])

results["date"] = pd.to_datetime(results["date"])

for _, row in former_names.iterrows():
    mask = (
        (results["date"] >= row["start_date"]) &
        (results["date"] <= row["end_date"])
    )

    results.loc[
        mask & (results["home_team"] == row["former"]),
        "home_team"
    ] = row["current"]

    results.loc[
        mask & (results["away_team"] == row["former"]),
        "away_team"
    ] = row["current"]

    results.loc[
        mask & (results["winner"] == row["former"]),
        "winner"
    ] = row["current"]

print("Normalización completada.")
print(results.shape)

display(results.head())

Normalización completada.
(49495, 11)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,winner,went_to_shootout
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,No Shootout,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,No Shootout,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,No Shootout,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,No Shootout,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,No Shootout,0


In [38]:
# ==========================================================
# Convertiremos nuevamente la fecha a formato YYYY-MM-DD y
# guardaremos el dataset histórico limpio.
# ==========================================================

results["date"] = results["date"].dt.strftime("%Y-%m-%d")

output_path = "../data/processed/results_historical_clean.csv"

results.to_csv(output_path, index=False)

print(f"Dataset guardado en: {output_path}")
print("Dimensiones finales:", results.shape)

Dataset guardado en: ../data/processed/results_historical_clean.csv
Dimensiones finales: (49495, 11)
